<a href="https://colab.research.google.com/github/maggiecrowner/DS5001-Final-Project/blob/main/Riffs2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Riff 2

In [1]:
! git clone https://github.com/maggiecrowner/DS5001-Final-Project.git

fatal: destination path 'DS5001-Final-Project' already exists and is not an empty directory.


In [2]:
!wget -O CORPUS.csv "https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv"
!wget -O LIB.csv "https://virginia.box.com/shared/static/fhzudg34je9xls5bfcbi4xdnaiek74rj.csv"
!wget -O DOC_SENT.csv "https://virginia.box.com/shared/static/g5r3ewbexgjc13pu1zqm21v3rpbuy5pb.csv"

--2026-04-21 00:34:40--  https://virginia.box.com/shared/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.box.com (virginia.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.box.com (virginia.box.com)|74.112.186.157|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-21 00:34:41--  https://virginia.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Reusing existing connection to virginia.box.com:443.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv [following]
--2026-04-21 00:34:41--  https://virginia.app.box.com/public/static/ijkqovrdgvrmctsdqobymig2p98q9x0m.csv
Resolving virginia.app.box.com (virginia.app.box.com)... 74.112.186.157, 2620:117:bff0:12d::
Connecting to virginia.app.box.com (virginia.app.box.com)|74.112.186.157|:443.

In [3]:
import pandas as pd
CORPUS = pd.read_csv('CORPUS.csv', delimiter='|', index_col=[0,1,2,3])
LIB = pd.read_csv('LIB.csv', delimiter='|', index_col=0)
DOC_SENT = pd.read_csv('DOC_SENT.csv', delimiter='|', index_col=0)

In [4]:
! pip install kaleido==0.2.1

## Scatter plot of avg song length vs avg sentiment per artist

In [6]:
import plotly.express as px
import plotly.io as pio
pio.renderers.default = 'colab'
import kaleido

merged = DOC_SENT.merge(
    LIB[['Artist', 'Album', 'Title', 'doc_length_words']],
    on=['Artist', 'Album', 'Title'],
    how='inner'
)

artist_df = merged.groupby('Artist').agg(
    avg_sentiment = ('sentiment_norm', 'mean'),
    avg_length    = ('doc_length_words', 'mean'),
    song_count    = ('doc_length_words', 'count')
).reset_index()

fig = px.scatter(
    artist_df,
    x='avg_length',
    y='avg_sentiment',
    text='Artist',
    size='song_count',
    color='avg_sentiment',
    color_continuous_scale='RdYlGn',
    title='Average Song Length vs. Average Sentiment per Artist',
    labels={
        'avg_length':    'Avg Song Length (words)',
        'avg_sentiment': 'Avg AFINN Sentiment (normalized)',
        'song_count':    'Song Count'
    },
    height=700,
    width=950
)

fig.update_traces(
    textposition='top center',
    textfont=dict(size=11)
)

fig.add_hline(y=0, line_dash='dash', line_color='grey',
              annotation_text='Neutral', annotation_position='right')

import time
time.sleep(2)
pio.write_image(fig, 'scatter_length_sentiment.png', format='png', scale=2, engine='kaleido')
fig.show()